# ROCKET Baseline — NFC Relay Detection (random split vs. Leave-One-Tag-Out)

Baseline for the NFC ATQA dataset produced by `wav_to_tabular.py`. Reads the per-label split CSVs
(`nfc_atqa_tag1_normal.csv` … `nfc_atqa_wireless_relay.csv`) or a single combined CSV.

**Why this notebook runs two evaluations**

Your CNN results show the tension that matters for a security write-up:

| Evaluation | CNN accuracy |
|---|---|
| Random / standard split | ~99.99% |
| Leave-One-Tag-Out (LOTO) | ~65.2% (wired-relay F1 ≈ 0 on 3 of 4 held-out tags) |

A random split lets samples from the *same physical tag* land in both train and test, so the model can win by memorizing each tag's hardware fingerprint instead of learning what a relay does. LOTO removes that shortcut: train on 3 tags, test on the 4th unseen tag. The collapse means the detector does **not** generalize to a tag it has never seen — and in the combined confusion matrix, thousands of wired-relay samples get accepted as `normal` (a relay attack passing as legitimate).

This notebook reproduces **both** evaluations with a ROCKET-family model, so you have a second, independent model telling the same story. Results are written in the same schema as `loto_cnn_results.csv` for a direct side-by-side.

> Uses **MiniRocket** by default — same random-kernel idea as ROCKET, but fast and memory-light enough for ~66k samples. Set `USE_MINIROCKET = False` to use plain Rocket.

## 1. Setup

In [ ]:
# Colab: %pip install aeon scikit-learn numpy pandas matplotlib
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import RidgeClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

USE_MINIROCKET = True
if USE_MINIROCKET:
    from aeon.transformations.collection.convolution_based import MiniRocket as RocketTransform
else:
    from aeon.transformations.collection.convolution_based import Rocket as RocketTransform
print("Transform:", RocketTransform.__name__)

## 2. Config

`DATA_GLOB` matches your split files. It also works if you point it at a single combined CSV.

In [ ]:
DATA_GLOB     = "data/nfc_atqa_*.csv"   # split files; or "data/nfc_atqa.csv" for the combined file
CLASS_COLUMN  = "class"                  # 3-class target: normal / wired_relay / wireless_relay
TAG_COLUMN    = "physical_tag"           # tag1..tag4  (used for LOTO)
INDEX_COLUMN  = "sample_index"           # used to spread wireless_relay across LOTO folds

CLASSES       = ["normal", "wired_relay", "wireless_relay"]
TAGS          = ["tag1", "tag2", "tag3", "tag4"]
N_RANDOM_FOLDS = 5
RANDOM_STATE  = 42
MAX_PER_CLASS = None     # set e.g. 4000 if RAM is tight; None = use everything

## 3. Load the split CSVs

Each file carries the metadata columns plus `x0…x1799`. We concatenate them; the signal is the `x*`
columns, the target is `class`, and `physical_tag` / `sample_index` drive the LOTO split.

In [ ]:
files = sorted(glob.glob(DATA_GLOB))
assert files, f"No files matched {DATA_GLOB}"
print(f"Loading {len(files)} file(s):")
for f in files: print("  ", os.path.basename(f))

df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

if MAX_PER_CLASS:
    df = (df.groupby(CLASS_COLUMN, group_keys=False)
            .apply(lambda g: g.sample(min(len(g), MAX_PER_CLASS), random_state=RANDOM_STATE)))

sig_cols = [c for c in df.columns if c.startswith("x") and c[1:].isdigit()]
sig_cols = sorted(sig_cols, key=lambda c: int(c[1:]))
print(f"\n{len(df)} samples, {len(sig_cols)} signal points each")
print(df[CLASS_COLUMN].value_counts())

X = df[sig_cols].to_numpy(np.float32)[:, np.newaxis, :]   # (N, 1, 1800)
y = df[CLASS_COLUMN].to_numpy()
tags = df[TAG_COLUMN].to_numpy()
sidx = df[INDEX_COLUMN].to_numpy()

## 4. Sanity check — one waveform per class

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for cls in CLASSES:
    i = np.where(y == cls)[0][0]
    ax.plot(X[i, 0], lw=0.8, label=cls)
ax.set_title("One ATQA waveform per class"); ax.set_xlabel("sample point"); ax.legend()
plt.tight_layout(); plt.show()

## 5. Shared fit/predict helper

Fits the random kernels on the training fold only, standardizes features with training statistics, then
fits a ridge classifier. The kernels never see test data.

In [ ]:
def fit_predict(Xtr, ytr, Xte):
    rk = RocketTransform(random_state=RANDOM_STATE)
    Ftr = np.asarray(rk.fit_transform(Xtr))
    Fte = np.asarray(rk.transform(Xte))
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-8
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit((Ftr - mu) / sd, ytr)
    return clf.predict((Fte - mu) / sd)

def per_class_f1_row(fold_name, y_true, y_pred):
    f1s = f1_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    return {"fold": fold_name,
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, labels=CLASSES, average="macro", zero_division=0),
            "normal_f1": f1s[0], "wired_relay_f1": f1s[1], "wireless_relay_f1": f1s[2]}

## 6. Evaluation A — random stratified split

This is the optimistic setting that matches the paper and your CNN's ~99.99%. Same physical tags appear
in train and test, so high accuracy here is expected — and is exactly the number LOTO will deflate.

In [ ]:
skf = StratifiedKFold(n_splits=N_RANDOM_FOLDS, shuffle=True, random_state=RANDOM_STATE)
rand_rows = []
for k, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y), 1):
    pred = fit_predict(X[tr], y[tr], X[te])
    rand_rows.append(per_class_f1_row(f"fold{k}", y[te], pred))
    print(f"fold {k}: acc = {rand_rows[-1]['accuracy']*100:.2f}%")

rand_df = pd.DataFrame(rand_rows)
print(f"\nRandom-split mean accuracy: {rand_df['accuracy'].mean()*100:.2f}% "
      f"(+/- {rand_df['accuracy'].std()*100:.2f})")

## 7. Evaluation B — Leave-One-Tag-Out (the honest test)

For each held-out tag, **all** of that tag's `normal` and `wired_relay` samples go to test; the model
trains on the other three tags. `wireless_relay` comes from a single emulator (not one of the 4 tags),
so it can't be held out by tag — we spread it deterministically across folds via `sample_index % 4`, so
it appears in both train and test in every fold. This isolates the real question: *does relay detection
generalize to a physical tag the model has never seen?*

In [ ]:
# fold key: per-tag classes -> their physical tag; wireless -> spread across the 4 folds
fold_key = np.where(
    np.isin(y, ["normal", "wired_relay"]),
    tags,
    np.array([f"tag{(int(s) % 4) + 1}" for s in sidx])
)

loto_rows = []
all_true, all_pred = [], []
for t in TAGS:
    te = fold_key == t
    tr = ~te
    pred = fit_predict(X[tr], y[tr], X[te])
    loto_rows.append(per_class_f1_row(t, y[te], pred))
    all_true.append(y[te]); all_pred.append(pred)
    r = loto_rows[-1]
    print(f"{t}: acc={r['accuracy']*100:5.2f}%  wired_relay_f1={r['wired_relay_f1']:.3f}")

loto_df = pd.DataFrame(loto_rows)
print(f"\nLOTO mean accuracy: {loto_df['accuracy'].mean()*100:.2f}% "
      f"(+/- {loto_df['accuracy'].std()*100:.2f})")
loto_df

### Combined LOTO confusion matrix

Pooled over all four held-out folds — directly comparable to your CNN confusion-matrix PNG.

In [ ]:
yt = np.concatenate(all_true); yp = np.concatenate(all_pred)
cm = confusion_matrix(yt, yp, labels=CLASSES)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=True)
ax.set_title("ROCKET LOTO — all folds combined"); plt.tight_layout(); plt.show()

# The security-critical number: relays accepted as normal
relay_as_normal = cm[CLASSES.index("wired_relay"), CLASSES.index("normal")]                 + cm[CLASSES.index("wireless_relay"), CLASSES.index("normal")]
print(f"Relay samples accepted as NORMAL under LOTO: {relay_as_normal}")

## 8. Binary view — normal vs. relay

Collapses both relay types into one attack class for the detection-oriented framing, under LOTO.

In [ ]:
to_bin = lambda a: np.where(a == "normal", "normal", "relay")
ytb, ypb = to_bin(yt), to_bin(yp)
print(f"Binary LOTO accuracy: {accuracy_score(ytb, ypb)*100:.2f}%\n")
print(classification_report(ytb, ypb, labels=["normal", "relay"], digits=4, zero_division=0))

## 9. Save results + compare to CNN

Writes `rocket_loto_results.csv` in the same columns as `loto_cnn_results.csv` so you can stack the two
models in one table.

In [ ]:
loto_df.to_csv("rocket_loto_results.csv", index=False)
rand_df.to_csv("rocket_random_results.csv", index=False)
print("Saved rocket_loto_results.csv and rocket_random_results.csv\n")

print("ROCKET-family summary")
print(f"  Random split : {rand_df['accuracy'].mean()*100:.2f}%")
print(f"  LOTO         : {loto_df['accuracy'].mean()*100:.2f}%")
print("\nIf you have loto_cnn_results.csv alongside, compare per-fold:")
print("  cnn = pd.read_csv('loto_cnn_results.csv')")
print("  pd.merge(cnn, loto_df, on='fold', suffixes=('_cnn','_rocket'))")

## 10. What to take away

- **The headline isn't the accuracy number — it's the gap.** A high random-split score and a low LOTO
  score together are the result: they show the model is keying on tag identity, not relay behavior.
- **Watch `wired_relay_f1` per fold.** If it craters on held-out tags (as in the CNN run), the detector
  can't recognize a relayed signal from a tag it hasn't trained on — the realistic deployment case.
- **Report the relay-accepted-as-normal count** from the LOTO confusion matrix. For a security audience
  that false-accept number matters more than overall accuracy.
- **Two models, one conclusion.** ROCKET and the CNN reaching the same random-vs-LOTO pattern is stronger
  evidence than either alone that the effect is about the data/evaluation, not one architecture.